In [1]:
import torch
import os
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image

In [2]:
class ImageProcessor:
    def __init__(self, root_dir, transform=None):
        self.root_dir = root_dir
        self.transform = transform

          
        self.image_paths = [os.path.join(root_dir, img) for img in os.listdir(root_dir)]
        

    def __len__(self):
        return len(self.image_paths)
    
    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        image = Image.open(img_path).convert('RGB')
        if self.transform:
            image = self.transform(image)
        return image
    
root_dir = './data./img_align_celeba/img_align_celeba'


In [3]:
transforms = transforms.Compose([
    transforms.CenterCrop(178),
    transforms.Resize(64),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

In [4]:
Dataset = ImageProcessor(root_dir, transform=transforms)
print(len(Dataset))

202599


In [5]:
DataLoader = DataLoader(Dataset, batch_size=128, shuffle=True)

Generator Network

In [6]:
import torch.nn as nn
import torch.optim as optim
import numpy as np

In [7]:
class Generator(torch.nn.Module):
    def __init__(self, z_dim = 100, img_channels = 3):
        super(Generator, self).__init__()

        self.model = nn.Sequential(
            nn.Linear(z_dim, 256),
            nn.ReLU(),

            nn.Linear(256, 512),
            nn.ReLU(),

            nn.Linear(512, 1024),
            nn.ReLU(),

            nn.Linear(1024, img_channels * 64 * 64),
            nn.Tanh()
        )


    def forward(self, z):
        img = self.model(z)
        img = img.view(img.size(0), 3, 64, 64)
        return img

In [ ]:
class Discriminator(torch.nn.Module):
    def __init__(self, img_channels = 3):
        super(Discriminator, self).__init__()

        self.model = nn.Sequential(
            nn.Flatten(),

            nn.Linear(img_channels * 64 * 64, 1024),
            nn.LeakyReLU(0.2, inplace=True),

            nn.Linear(1024, 512),
            nn.LeakyReLU(0.2, inplace=True),

            nn.Linear(512, 256),            
            nn.LeakyReLU(0.2, inplace=True),

            nn.Linear(256, 1),
            nn.Sigmoid()

        )

    def forward(self, img):
        img_flat = img.view(img.size(0), -1)
        validity = self.model(img_flat)
        return validity
    
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

generator = Generator().to(device)
discriminator = Discriminator().to(device)  
print(device)


Using device: cuda
cuda


# Loss and optimizers

In [9]:
GAN_loss = nn.BCELoss()

optimizer_G = optim.Adam(generator.parameters(), lr=0.0002, betas=(0.5, 0.999))
optimizer_D = optim.Adam(discriminator.parameters(), lr=0.0002, betas=(0.5, 0.999))

Training GAN

In [10]:
def train(generator, discriminator, dataloader, epochs=50):
    for epoch in range(epochs):
        for i, imgs in enumerate(dataloader):
            batch_size = imgs.size(0)
            real_imgs = imgs.to(device)

            real_labels = torch.ones(batch_size, 1).to(device)
            fake_labels = torch.zeros(batch_size, 1).to(device)

            # Train Discriminator
            optimizer_D.zero_grad()

            output = discriminator(real_imgs)
            d_loss_real = GAN_loss(output, real_labels)
            d_loss_real.backward()

            z = torch.randn(batch_size, 100).to(device)

            fake_imgs = generator(z)
            output = discriminator(fake_imgs)
            d_loss_fake = GAN_loss(output, fake_labels)
            d_loss_fake.backward()


            d_loss = d_loss_real + d_loss_fake
            optimizer_D.step()

            # Train Generator


            optimizer_G.zero_grad()
            z = torch.randn(batch_size, 100).to(device)

            fake_imgs = generator(z)
            output = discriminator(fake_imgs)
            g_loss = GAN_loss(output, real_labels)
            g_loss.backward()
            optimizer_G.step()

            if i % 50 == 0:
                print(f"Epoch [{epoch}/{epochs}] Batch {i}/{len(dataloader)} \
                      Loss D: {d_loss.item():.4f}, loss G: {g_loss.item():.4f}")




In [11]:
# save gen and disc images

import matplotlib.pyplot as plt
import torchvision.utils as vutils
import numpy as np

def save_generated_images(generator, epoch, num_images=16):

    z = torch.randn(num_images, 100).to(device)
    fake_imgs = generator(z)
    fake_imgs = fake_imgs.cpu().detach()

    grid = vutils.make_grid(fake_imgs, padding=2, normalize=True)
    plt.figure(figsize=(8, 8))
    plt.axis("off")
    plt.title(f"Generated Images at Epoch {epoch}")
    plt.imshow(np.transpose(grid, (1, 2, 0)))
    plt.savefig(f"generated_images_epoch_{epoch}.png")
    plt.close()


In [33]:
train(generator, discriminator, DataLoader, epochs=50)

Epoch [0/50] Batch 0/1583                       Loss D: 1.3902, loss G: 0.6830
Epoch [0/50] Batch 50/1583                       Loss D: 0.6725, loss G: 0.9943
Epoch [0/50] Batch 100/1583                       Loss D: 1.4435, loss G: 0.4771
Epoch [0/50] Batch 150/1583                       Loss D: 0.4684, loss G: 1.1240
Epoch [0/50] Batch 200/1583                       Loss D: 0.2920, loss G: 1.7676
Epoch [0/50] Batch 250/1583                       Loss D: 0.2960, loss G: 2.3633
Epoch [0/50] Batch 300/1583                       Loss D: 0.6674, loss G: 0.1874
Epoch [0/50] Batch 350/1583                       Loss D: 0.1801, loss G: 3.5081
Epoch [0/50] Batch 400/1583                       Loss D: 0.1467, loss G: 3.0162
Epoch [0/50] Batch 450/1583                       Loss D: 0.1166, loss G: 2.8485
Epoch [0/50] Batch 500/1583                       Loss D: 0.1419, loss G: 2.7910
Epoch [0/50] Batch 550/1583                       Loss D: 0.1954, loss G: 3.9714
Epoch [0/50] Batch 600/1583    

KeyboardInterrupt: 